# Задача 5: Трансформеры (RuBERT)
Обучение модели RuBERT-Tiny. Мы используем сырой текст (CONTENT) без лемматизации, так как BERT сам отлично понимает контекст.

In [1]:
import sys
import os
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
from sklearn.metrics import f1_score, classification_report

# Загрузка данных
train_df = pd.read_parquet('../data/train.parquet')
test_df = pd.read_parquet('../data/test.parquet')

# БЕРЕМ СЫРОЙ ТЕКСТ (CONTENT)
train_texts = train_df['CONTENT'].astype(str).tolist()
test_texts = test_df['CONTENT'].astype(str).tolist()

# ВАЖНО: Нейросети требуют метки от 0 до N-1. 
# У нас классы от 1 до 8 (но 7 отсутствует). Сдвигаем их на -1.
train_labels = [int(label) - 1 for label in train_df['class']]
test_labels = [int(label) - 1 for label in test_df['class']]

# Количество уникальных классов (учитывая, что 8 -> 7)
num_labels = 8 

print(f"Данные для BERT готовы. Тренировка: {len(train_texts)}, Тест: {len(test_texts)}")

c:\Users\Nil\OneDrive\Desktop\labs\anpyt\lab6\lab6\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Данные для BERT готовы. Тренировка: 529, Тест: 59


## Загрузка модели и токенизация
Используем легкую версию rubert-tiny2 для быстрого обучения.

In [2]:
model_name = "cointegrated/rubert-tiny2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

train_dataset = Dataset.from_dict({"text": train_texts, "label": train_labels})
test_dataset = Dataset.from_dict({"text": test_texts, "label": test_labels})

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=256)

train_tokenized = train_dataset.map(tokenize_function, batched=True)
test_tokenized = test_dataset.map(tokenize_function, batched=True)

c:\Users\Nil\OneDrive\Desktop\labs\anpyt\lab6\lab6\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Nil\.cache\huggingface\hub\models--cointegrated--rubert-tiny2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\Nil\OneDrive\Desktop\labs\anpyt\lab6\lab6\.venv\Lib\site-packages\huggingface_h

## Обучение и оценка

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"macro_f1": f1_score(labels, predictions, average="macro")}

training_args = TrainingArguments(
    output_dir="./results_rubert",
    num_train_epochs=10,            # <--- УВЕЛИЧИВАЕМ ДО 10 (пусть учится дольше)
    per_device_train_batch_size=8,  # <--- УМЕНЬШАЕМ БАТЧ (больше шагов обновления)
    per_device_eval_batch_size=16,
    learning_rate=5e-5,             # <--- Чуть увеличиваем скорость обучения
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    logging_steps=10,               # Чтобы видеть прогресс каждые 10 шагов
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    compute_metrics=compute_metrics,
)

print("Начинаем обучение BERT...")
trainer.train()

Начинаем обучение BERT...


c:\Users\Nil\OneDrive\Desktop\labs\anpyt\lab6\lab6\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Macro F1
1,No log,1.600982,0.094156
2,No log,1.520738,0.094156


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  6.59it/s]
c:\Users\Nil\OneDrive\Desktop\labs\anpyt\lab6\lab6\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  7.13it/s]


TrainOutput(global_step=68, training_loss=1.6857220144832836, metrics={'train_runtime': 83.2507, 'train_samples_per_second': 12.709, 'train_steps_per_second': 0.817, 'total_flos': 3904007405568.0, 'train_loss': 1.6857220144832836, 'epoch': 2.0})

## Финальная оценка

In [ ]:
results = trainer.evaluate()
print(f"\nMacro F1 (RuBERT-Tiny): {results['eval_macro_f1']:.4f}")

# Предсказания для отчета
predictions = trainer.predict(test_tokenized)
y_pred_bert = np.argmax(predictions.predictions, axis=-1)

print(classification_report(test_labels, y_pred_bert))

# Сохранение
results_df = pd.DataFrame({
    'Model': ['RuBERT-Tiny'],
    'Macro_F1': [results['eval_macro_f1']]
})
os.makedirs('../results', exist_ok=True)
results_df.to_csv('../results/transformer_results.csv', index=False)
print("Результаты сохранены!")